# Patched Neural Field Diffusion on Synthetic Data

Test the new `PatchedNeuralFieldDiffusion` architecture on synthetic toy data.

**Key Innovation**: Space-filling curve serialization + per-patch neural fields
- Morton curve preserves 3D locality in 1D ordering
- Each patch (16 points) gets its own context → its own MLP
- Distance-weighted blending for smooth transitions

This should solve the global pooling collapse problem and enable learning
multi-modal geometry (multi-sphere, chairs with separate parts).

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from tqdm.notebook import tqdm
import time

# Our modules
from src.models.patched_neural_field import PatchedNeuralFieldDiffusion
from src.models.neural_field import NeuralFieldDiffusion
from src.diffusion.flow_matching import FlowMatchingLoss, FlowMatchingSampler
from data.toy_data import (
    generate_torus, generate_sphere, generate_helix,
    generate_multi_sphere_cube, generate_multi_sphere_ring,
    generate_two_spheres, generate_four_spheres,
    get_all_generators
)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## 1. Configuration

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Data
SHAPES = ['multi_sphere_ring']  # Try: 'two_spheres', 'four_spheres', 'multi_sphere_cube', 'torus'
PATCH_SIZE = 16
N_POINTS = 512                  # Must be divisible by PATCH_SIZE
N_SAMPLES = 1000                # Training samples

# Model
HIDDEN_SIZE = 128
HIDDEN_SIZE_X = 32
NUM_HEADS = 4
NUM_BLOCKS = 4
NUM_NERF_BLOCKS = 2
NERF_MLP_RATIO = 2
MAX_FREQS = 6
BLEND_TEMPERATURE = 0.1

# Training
EPOCHS = 300
BATCH_SIZE = 32
LR = 1e-4

# Verify
assert N_POINTS % PATCH_SIZE == 0, f"N_POINTS must be divisible by PATCH_SIZE"
print(f"Shapes: {SHAPES}")
print(f"Points: {N_POINTS}, Patches: {N_POINTS // PATCH_SIZE} x {PATCH_SIZE}")

## 2. Dataset

In [ ]:
class ToyDataset(torch.utils.data.Dataset):
    """Simple toy dataset with pre-generated samples."""
    
    def __init__(self, shapes, n_points, n_samples):
        self.generators = get_all_generators()
        self.shape_names = shapes
        self.n_points = n_points
        
        # Pre-generate samples
        self.samples = []
        samples_per_shape = n_samples // len(shapes)
        
        for shape_name in shapes:
            gen_func = self.generators[shape_name]
            for _ in range(samples_per_shape):
                pc = gen_func(n_points)
                pc = pc.normalize()
                self.samples.append(torch.tensor(pc.points, dtype=torch.float32))
        
        print(f"Generated {len(self.samples)} samples")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]


# Create dataset
dataset = ToyDataset(SHAPES, N_POINTS, N_SAMPLES)
dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True
)

In [ ]:
# Visualize training samples
fig = plt.figure(figsize=(16, 4))

for i in range(4):
    sample = dataset[i * 100].numpy()
    
    ax = fig.add_subplot(1, 4, i + 1, projection='3d')
    colors = sample[:, 2]
    ax.scatter(sample[:, 0], sample[:, 1], sample[:, 2],
               c=colors, cmap='viridis', s=5, alpha=0.7)
    ax.set_title(f'Sample {i+1}')
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])

plt.suptitle(f'Training Samples: {SHAPES}', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Create Models (Patched vs Original)

In [ ]:
# Patched model (NEW)
patched_model = PatchedNeuralFieldDiffusion(
    in_channels=3,
    out_channels=3,
    hidden_size=HIDDEN_SIZE,
    hidden_size_x=HIDDEN_SIZE_X,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    num_nerf_blocks=NUM_NERF_BLOCKS,
    nerf_mlp_ratio=NERF_MLP_RATIO,
    max_freqs=MAX_FREQS,
    patch_size=PATCH_SIZE,
    blend_temperature=BLEND_TEMPERATURE,
).to(DEVICE)

# Original model for comparison
original_model = NeuralFieldDiffusion(
    in_channels=3,
    out_channels=3,
    hidden_size=HIDDEN_SIZE,
    hidden_size_x=HIDDEN_SIZE_X,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    num_cond_blocks=NUM_BLOCKS // 2,
    nerf_mlp_ratio=NERF_MLP_RATIO,
    max_freqs=MAX_FREQS,
).to(DEVICE)

print(f"Patched model: {sum(p.numel() for p in patched_model.parameters()):,} params")
print(f"Original model: {sum(p.numel() for p in original_model.parameters()):,} params")

# Test forward pass
with torch.no_grad():
    test_x = torch.randn(2, N_POINTS, 3, device=DEVICE)
    test_t = torch.rand(2, device=DEVICE)
    
    out_patched = patched_model(test_x, test_t)
    out_original = original_model(test_x, test_t)
    
    print(f"\nPatched output: {out_patched.shape}")
    print(f"Original output: {out_original.shape}")

## 4. Training

In [ ]:
def train_model(model, dataloader, epochs, lr, device, name="Model"):
    """Train a model and return loss history."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    loss_fn = FlowMatchingLoss(schedule_type='linear')
    
    losses = []
    pbar = tqdm(range(epochs), desc=name)
    
    for epoch in pbar:
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        
        for batch in dataloader:
            x0 = batch.to(device)
            
            optimizer.zero_grad()
            output = loss_fn(model, x0)
            loss = output['loss']
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        scheduler.step()
        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)
        
        pbar.set_postfix({'loss': f'{avg_loss:.4f}'})
    
    return losses

In [ ]:
# Train PATCHED model
print("="*60)
print("Training PATCHED model...")
print("="*60)
patched_losses = train_model(patched_model, dataloader, EPOCHS, LR, DEVICE, "Patched")

In [ ]:
# Train ORIGINAL model for comparison
print("="*60)
print("Training ORIGINAL model...")
print("="*60)
original_losses = train_model(original_model, dataloader, EPOCHS, LR, DEVICE, "Original")

In [ ]:
# Compare training curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(patched_losses, label='Patched', color='blue')
plt.plot(original_losses, label='Original', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Comparison')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(patched_losses[20:], label='Patched', color='blue')
plt.plot(original_losses[20:], label='Original', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss (after warmup)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"\nFinal Loss - Patched: {patched_losses[-1]:.4f}, Original: {original_losses[-1]:.4f}")

## 5. Generate Samples

In [ ]:
def generate_samples(model, n_samples=4, n_points=512, n_steps=100, device='cpu'):
    """Generate samples using Euler integration."""
    model.eval()
    sampler = FlowMatchingSampler(model)
    noise = torch.randn(n_samples, n_points, 3, device=device)
    
    with torch.no_grad():
        samples = sampler.sample_euler(noise, n_steps=n_steps)
    
    return samples.cpu().numpy()

In [ ]:
# Generate from both models
print("Generating samples...")
patched_samples = generate_samples(patched_model, n_samples=4, n_points=N_POINTS, device=DEVICE)
original_samples = generate_samples(original_model, n_samples=4, n_points=N_POINTS, device=DEVICE)
print("Done!")

In [ ]:
# Compare generated samples
fig = plt.figure(figsize=(16, 12))

# Row 1: Ground Truth
for i in range(4):
    gt = dataset[i * 100].numpy()
    ax = fig.add_subplot(3, 4, i + 1, projection='3d')
    colors = gt[:, 2]
    ax.scatter(gt[:, 0], gt[:, 1], gt[:, 2], c=colors, cmap='viridis', s=3, alpha=0.7)
    ax.set_title(f'GT {i+1}')
    ax.set_xlim([-1.5, 1.5])
    ax.set_ylim([-1.5, 1.5])
    ax.set_zlim([-1.5, 1.5])

# Row 2: Patched Model
for i in range(4):
    sample = patched_samples[i]
    ax = fig.add_subplot(3, 4, i + 5, projection='3d')
    colors = sample[:, 2]
    ax.scatter(sample[:, 0], sample[:, 1], sample[:, 2], c=colors, cmap='plasma', s=3, alpha=0.7)
    ax.set_title(f'Patched {i+1}')
    ax.set_xlim([-1.5, 1.5])
    ax.set_ylim([-1.5, 1.5])
    ax.set_zlim([-1.5, 1.5])

# Row 3: Original Model
for i in range(4):
    sample = original_samples[i]
    ax = fig.add_subplot(3, 4, i + 9, projection='3d')
    colors = sample[:, 2]
    ax.scatter(sample[:, 0], sample[:, 1], sample[:, 2], c=colors, cmap='coolwarm', s=3, alpha=0.7)
    ax.set_title(f'Original {i+1}')
    ax.set_xlim([-1.5, 1.5])
    ax.set_ylim([-1.5, 1.5])
    ax.set_zlim([-1.5, 1.5])

plt.suptitle(f'{SHAPES}: GT (top) vs Patched (middle) vs Original (bottom)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Test Different Shapes

In [ ]:
# Quick test on simpler shapes to verify the architecture works
test_shapes = ['two_spheres', 'four_spheres', 'torus']

fig = plt.figure(figsize=(12, 4))

generators = get_all_generators()
for i, shape_name in enumerate(test_shapes):
    gen_func = generators[shape_name]
    pc = gen_func(N_POINTS)
    pc = pc.normalize()
    points = pc.points
    
    ax = fig.add_subplot(1, 3, i + 1, projection='3d')
    colors = points[:, 2]
    ax.scatter(points[:, 0], points[:, 1], points[:, 2], c=colors, cmap='viridis', s=3, alpha=0.7)
    ax.set_title(shape_name)
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])

plt.suptitle('Test Shapes for Debugging', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Visualize Space-Filling Curve Ordering

In [ ]:
from src.models.patched_neural_field import sort_by_space_filling_curve, patchify_points

# Visualize how Morton curve orders points
sample = dataset[0].unsqueeze(0)  # [1, N, 3]
sorted_sample, indices = sort_by_space_filling_curve(sample)
patches, centers = patchify_points(sorted_sample, PATCH_SIZE)

fig = plt.figure(figsize=(16, 4))

# Original order
ax1 = fig.add_subplot(1, 4, 1, projection='3d')
points = sample[0].numpy()
colors = np.arange(len(points))  # Color by original index
ax1.scatter(points[:, 0], points[:, 1], points[:, 2], c=colors, cmap='viridis', s=5)
ax1.set_title('Original Order')

# Morton-sorted order
ax2 = fig.add_subplot(1, 4, 2, projection='3d')
points = sorted_sample[0].numpy()
colors = np.arange(len(points))  # Color by sorted index
ax2.scatter(points[:, 0], points[:, 1], points[:, 2], c=colors, cmap='viridis', s=5)
ax2.set_title('Morton Curve Order')

# Patches colored
ax3 = fig.add_subplot(1, 4, 3, projection='3d')
patch_colors = np.repeat(np.arange(patches.shape[1]), PATCH_SIZE)
ax3.scatter(points[:, 0], points[:, 1], points[:, 2], c=patch_colors, cmap='tab20', s=5)
ax3.set_title(f'Patches ({patches.shape[1]} patches)')

# Patch centers
ax4 = fig.add_subplot(1, 4, 4, projection='3d')
ax4.scatter(points[:, 0], points[:, 1], points[:, 2], c='lightgray', s=2, alpha=0.3)
center_pts = centers[0].numpy()
ax4.scatter(center_pts[:, 0], center_pts[:, 1], center_pts[:, 2], c='red', s=50, marker='o')
ax4.set_title('Patch Centers (red)')

plt.suptitle('Space-Filling Curve Visualization', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Points: {sample.shape[1]}, Patches: {patches.shape[1]}, Patch size: {PATCH_SIZE}")

## 8. Summary

### Expected Results

**If Patched model works better:**
- Loss should decrease below 0.2 (vs ~0.4 for original)
- Generated samples should show distinct spheres/parts
- Multi-sphere should look like multiple spheres, not a blob

**If both models fail:**
- Try simpler shapes first: `two_spheres` → `four_spheres` → `multi_sphere_ring`
- May need more epochs or larger model

### Key Differences

| Aspect | Original | Patched |
|--------|----------|----------|
| Context | Global mean pooling | Per-patch context |
| MLP | Same for all points | Different per patch |
| Locality | Lost | Preserved via Morton curve |
| Multi-modal | Collapses modes | Separates modes |